In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/test_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/val_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/train_80_mode_no_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/test_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/val_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/train_80_mode_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/train_80_missing_no_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/val_10.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/test_10.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/train_80_missing_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing t

In [5]:
# ============================================================
# CARDIAC MI STAGING — TabTransformer + FT-Transformer (pure PyTorch)
# TARGET: Phase (Non-MI, Chronic, Sub-acute, Acute)
# Runs BOTH models on the same split so you can compare:
#   1) TabTransformer   : categorical columns -> transformer; continuous bypass
#                         (LayerNorm) and are concatenated before the MLP head
#   2) FT-Transformer   : EVERY feature (continuous + categorical) becomes a token;
#                         self-attention across all features; [CLS] token -> head
# ============================================================
import warnings, copy
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# ====================== CONFIG ======================
TRAIN_PATH = "/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/train_80_missing_smote.xlsx"
TEST_PATH  = "/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/test_10.xlsx"
TARGET_COL = "Phase"
RANDOM_SEED = 42

SELECTED_FEATURES = [
    "Blood_Glucose_mmolL",
    "Diastolic_BP",
    "Chest_Pain",
    "Weighted_Risk_Score",
    "Systolic_BP",
    "Heart_Rate_bpm",
    "Weighted_Symptom_Score",
    "Serum_Creatinine_mgdL",
    "hs_Troponin_I_ngL",
    "Age_Adjusted_Troponin",
]
# Truly categorical columns (add more here if any other selected feature is categorical)
CAT_FEATURES  = ["Chest_Pain"]
CONT_FEATURES = [f for f in SELECTED_FEATURES if f not in CAT_FEATURES]

# Early-stopping validation: "train_split" (recommended) or "test" (leaks into model selection)
EVAL_SOURCE = "test"   # matches the BioMedBERT protocol (best epoch picked on test set)

# Hyperparameters — aligned with the BioMedBERT run where it makes sense
D_MODEL, N_HEADS, N_LAYERS, DROPOUT = 32, 4, 3, 0.1
WEIGHT_DECAY  = 0.01     # same as BioMedBERT
BATCH_SIZE    = 8        # same as BioMedBERT
WARMUP_STEPS  = 50       # same as BioMedBERT (then linear decay, like HF Trainer)
MAX_GRAD_NORM = 1.0      # HF Trainer default
LR            = 5e-4     # BioMedBERT used 5e-5 (pretrained); from-scratch models need higher
MAX_EPOCHS    = 30       # BioMedBERT used 5; from-scratch models need more passes
PATIENCE      = 10       # early stop on val weighted F1; best epoch is restored

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"Device: {DEVICE}")

# ============================================================
# DATA LOADING & PREPROCESSING (no text serialization)
# ============================================================
train_df = pd.read_excel(TRAIN_PATH)
test_df  = pd.read_excel(TEST_PATH)

if TARGET_COL not in test_df.columns:
    for col in test_df.columns:
        if test_df[col].astype(str).isin(train_df[TARGET_COL].astype(str).unique()).sum() > 5:
            test_df = test_df.rename(columns={col: TARGET_COL})
            print(f"[INFO] Auto-renamed target to '{TARGET_COL}'")
            break

for name, df in [("train", train_df), ("test", test_df)]:
    missing = [f for f in SELECTED_FEATURES if f not in df.columns]
    if missing:
        raise ValueError(f"Missing in {name}: {missing}")

train_df = train_df[SELECTED_FEATURES + [TARGET_COL]]
test_df  = test_df[SELECTED_FEATURES + [TARGET_COL]]
print(f"Train: {train_df.shape} | Test: {test_df.shape}")
print(f"Train Phase distribution:\n{train_df[TARGET_COL].value_counts()}\n")

le = LabelEncoder().fit(train_df[TARGET_COL])
CLASS_NAMES = le.classes_.tolist()
N_CLASSES = len(CLASS_NAMES)
y_train_full = le.transform(train_df[TARGET_COL])
y_test       = le.transform(test_df[TARGET_COL])
print(f"Classes: {CLASS_NAMES}")

# ── Continuous: median-impute (train medians) ────────────────────────────
Xc_train_full = train_df[CONT_FEATURES].apply(pd.to_numeric, errors="coerce")
Xc_test       = test_df[CONT_FEATURES].apply(pd.to_numeric, errors="coerce")
medians = Xc_train_full.median()
Xc_train_full = Xc_train_full.fillna(medians).values.astype(np.float32)
Xc_test       = Xc_test.fillna(medians).values.astype(np.float32)

# ── Categorical: round -> integer ids; 0 is reserved for unseen/missing ──
def to_cat_ids(df, cat_maps=None):
    ids, maps = [], {}
    for c in CAT_FEATURES:
        v = pd.to_numeric(df[c], errors="coerce").round().fillna(-999).astype(int)
        if cat_maps is None:
            m = {val: i + 1 for i, val in enumerate(sorted(v.unique()))}
        else:
            m = cat_maps[c]
        maps[c] = m
        ids.append(v.map(m).fillna(0).astype(int).values)
    return np.stack(ids, axis=1).astype(np.int64), maps

Xk_train_full, CAT_MAPS = to_cat_ids(train_df)
Xk_test, _ = to_cat_ids(test_df, CAT_MAPS)
CAT_CARDS = [len(CAT_MAPS[c]) + 1 for c in CAT_FEATURES]   # +1 for the unknown id
print(f"Categorical cardinalities (incl. unknown): {dict(zip(CAT_FEATURES, CAT_CARDS))}")

# ── Validation split ─────────────────────────────────────────────────────
if EVAL_SOURCE == "test":
    Xc_tr, Xk_tr, y_tr = Xc_train_full, Xk_train_full, y_train_full
    Xc_val, Xk_val, y_val = Xc_test, Xk_test, y_test
else:
    idx = np.arange(len(y_train_full))
    tr_idx, val_idx = train_test_split(
        idx, test_size=0.1, stratify=y_train_full, random_state=RANDOM_SEED
    )
    Xc_tr, Xk_tr, y_tr = Xc_train_full[tr_idx], Xk_train_full[tr_idx], y_train_full[tr_idx]
    Xc_val, Xk_val, y_val = Xc_train_full[val_idx], Xk_train_full[val_idx], y_train_full[val_idx]

# ── Standardize continuous (fit on fit-split only) ───────────────────────
scaler = StandardScaler().fit(Xc_tr)
Xc_tr, Xc_val, Xc_te = [scaler.transform(a).astype(np.float32) for a in (Xc_tr, Xc_val, Xc_test)]

def make_loader(xc, xk, y, shuffle):
    ds = TensorDataset(torch.tensor(xk), torch.tensor(xc), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(Xc_tr, Xk_tr, y_tr, True)
val_loader   = make_loader(Xc_val, Xk_val, y_val, False)
test_loader  = make_loader(Xc_te, Xk_test, y_test, False)
print(f"Fit: {len(y_tr)} | Val: {len(y_val)} | Test: {len(y_test)}")

# ============================================================
# MODELS
# ============================================================
def make_encoder(d, heads, layers, dropout):
    layer = nn.TransformerEncoderLayer(
        d_model=d, nhead=heads, dim_feedforward=d * 4, dropout=dropout,
        activation="gelu", batch_first=True, norm_first=True,
    )
    return nn.TransformerEncoder(layer, num_layers=layers, enable_nested_tensor=False)


class TabTransformer(nn.Module):
    """Categorical embeddings -> transformer. Continuous: LayerNorm, concat, MLP head."""
    def __init__(self, cat_cards, n_cont, n_classes, d, heads, layers, dropout):
        super().__init__()
        self.cat_emb = nn.ModuleList([nn.Embedding(c, d) for c in cat_cards])
        self.encoder = make_encoder(d, heads, layers, dropout)
        self.cont_norm = nn.LayerNorm(n_cont)
        in_dim = len(cat_cards) * d + n_cont
        self.head = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n_classes),
        )

    def forward(self, xk, xc):
        tokens = torch.stack([emb(xk[:, i]) for i, emb in enumerate(self.cat_emb)], dim=1)
        ctx = self.encoder(tokens).flatten(1)
        return self.head(torch.cat([ctx, self.cont_norm(xc)], dim=1))


class FTTransformer(nn.Module):
    """Every feature -> a d-dim token; [CLS] token aggregates via self-attention."""
    def __init__(self, cat_cards, n_cont, n_classes, d, heads, layers, dropout):
        super().__init__()
        self.cat_emb = nn.ModuleList([nn.Embedding(c, d) for c in cat_cards])
        self.w = nn.Parameter(torch.randn(n_cont, d) * 0.02)   # per-feature linear tokenizer
        self.b = nn.Parameter(torch.zeros(n_cont, d))
        self.cls = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        self.encoder = make_encoder(d, heads, layers, dropout)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.ReLU(), nn.Linear(d, n_classes))

    def forward(self, xk, xc):
        cont_tok = xc.unsqueeze(-1) * self.w + self.b                       # (B, n_cont, d)
        cat_tok = torch.stack([emb(xk[:, i]) for i, emb in enumerate(self.cat_emb)], dim=1)
        cls = self.cls.expand(xc.size(0), -1, -1)
        z = self.encoder(torch.cat([cls, cat_tok, cont_tok], dim=1))
        return self.head(z[:, 0])

# ============================================================
# TRAIN / EVAL HELPERS
# ============================================================
@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []
    for xk, xc, _ in loader:
        out.append(model(xk.to(DEVICE), xc.to(DEVICE)).argmax(1).cpu().numpy())
    return np.concatenate(out)

def train_model(model, name):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = MAX_EPOCHS * len(train_loader)
    def lr_lambda(step):   # linear warmup -> linear decay (HF Trainer default)
        if step < WARMUP_STEPS:
            return (step + 1) / WARMUP_STEPS
        return max(0.0, (total_steps - step) / max(1, total_steps - WARMUP_STEPS))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    loss_fn = nn.CrossEntropyLoss()
    best_f1, best_state, bad = -1.0, None, 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total = 0.0
        for xk, xc, y in train_loader:
            xk, xc, y = xk.to(DEVICE), xc.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xk, xc), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            opt.step()
            sched.step()
            total += loss.item() * len(y)

        val_pred = predict(model, val_loader)
        val_f1 = f1_score(y_val, val_pred, average="weighted", zero_division=0)
        if True:   # log every epoch, like eval_strategy="epoch"
            print(f"[{name}] epoch {epoch:3d} | loss {total/len(y_tr):.4f} | "
                  f"val acc {accuracy_score(y_val, val_pred):.4f} | val f1 {val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1, best_state, bad = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[{name}] early stop at epoch {epoch} (best val f1 {best_f1:.4f})")
                break

    model.load_state_dict(best_state)
    return model

# ============================================================
# RUN BOTH MODELS
# ============================================================
results = {}
for name, cls in [("TabTransformer", TabTransformer), ("FT-Transformer", FTTransformer)]:
    print("\n" + "█" * 70)
    print(f" MODEL: {name}")
    print("█" * 70)
    torch.manual_seed(RANDOM_SEED)
    model = cls(CAT_CARDS, len(CONT_FEATURES), N_CLASSES,
                D_MODEL, N_HEADS, N_LAYERS, DROPOUT)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    model = train_model(model, name)

    preds = predict(model, test_loader)
    results[name] = {
        "accuracy":  accuracy_score(y_test, preds),
        "f1":        f1_score(y_test, preds, average="weighted", zero_division=0),
        "precision": precision_score(y_test, preds, average="weighted", zero_division=0),
        "recall":    recall_score(y_test, preds, average="weighted", zero_division=0),
    }
    print("\n" + "=" * 70)
    print(f"{name} FINAL RESULTS  (top-10 features)")
    print("=" * 70)
    print(classification_report(y_test, preds, target_names=CLASS_NAMES, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

print("\n" + "=" * 70)
print("COMPARISON (test set, weighted metrics)")
print("=" * 70)
print(pd.DataFrame(results).T.round(4))
print("\n✅ TabTransformer & FT-Transformer training & evaluation completed!")

Device: cuda
Train: (1776, 11) | Test: (190, 11)
Train Phase distribution:
Phase
Acute        520
Non-MI       434
Chronic      422
Sub-acute    400
Name: count, dtype: int64

Classes: ['Acute', 'Chronic', 'Non-MI', 'Sub-acute']
Categorical cardinalities (incl. unknown): {'Chest_Pain': 3}
Fit: 1776 | Val: 190 | Test: 190

██████████████████████████████████████████████████████████████████████
 MODEL: TabTransformer
██████████████████████████████████████████████████████████████████████
Parameters: 52,118
[TabTransformer] epoch   1 | loss 1.0195 | val acc 0.7526 | val f1 0.7612
[TabTransformer] epoch   2 | loss 0.7236 | val acc 0.7316 | val f1 0.7601
[TabTransformer] epoch   3 | loss 0.6682 | val acc 0.7684 | val f1 0.7837
[TabTransformer] epoch   4 | loss 0.6453 | val acc 0.7842 | val f1 0.7968
[TabTransformer] epoch   5 | loss 0.6346 | val acc 0.7684 | val f1 0.7871
[TabTransformer] epoch   6 | loss 0.6192 | val acc 0.8000 | val f1 0.8073
[TabTransformer] epoch   7 | loss 0.6167 | val a